In [ ]:
# TODO -> extract out to requirements.txt
# TODO -> add better logging
!uv pip install pandas pyarrow polars snowflake-snowpark-python urllib3
!uv pip list

Resolved 37 packages in 348ms                                        
Installed 1 package in 15ms                                 
 + pyarrow==26.0.0
Package                    Version
-------------------------- -----------
appnope                    1.0.0
asn1crypto                 1.5.1
asttokens                  3.0.2
boto3                      1.43.110
botocore                   1.43.110
certifi                    2026.7.22
cffi                       2.1.1
charset-normalizer         3.5.2
cloudpickle                3.1.1
comm                       0.2.3
cryptography               50.0.2
debugpy                    1.8.22
executing                  2.2.1
filelock                   3.32.7
idna                       3.20
ipykernel                  7.4.0
ipython                    9.17.1
ipython-pygments-lexers    1.1.1
jedi                       0.20.0
jmespath                   1.1.0
jupyter-client             8.10.0
jupyter-core               5.9.1
matplotlib-inline          0.2.2
ne

In [11]:
import json           # working with json data
import os             # traverse dirs and get file names
import polars as pl   # memory-efficient dataframes
import shutil         # high-level file ops (i.e. write data from http req to file)
import urllib3        # requests lib with connection pooling and QoL features

# Snowpark is Snowflake's best practices lib for interfacing with
# snowflake from python, will use SSO for auth
from snowflake.snowpark.session import Session
from typing import Optional

In [36]:
# CONSTANTS
PRESIGNED_URLS_TXT_FILE_NAME = 'presign_1000'
APPROX_MANIFEST_COUNT = 2000
MARGIN_OF_ERROR = 250

TARGET_DIR = 'data'

TARGET_DATABASE = "ANALYTICS_DEV"
TARGET_SCHEMA = "DBT_KMCCLEARYSHARPE"
TARGET_TABLE = "MODEL_ID_TO_FQN_MAPPINGS_FROM_MANIFESTS"

SNOW_CONN_PARAMS = {
    "account": "kw27752",
    "user": "KEEGAN.MCCLEARYSHARPE",
    "authenticator": "externalbrowser",
    "database": {TARGET_DATABASE},
    "schema": {TARGET_SCHEMA},
    "warehouse": "TRANSFORMING", # small warehouse
    "role": "TRANSFORMER",       # typical role for tables comprised of transforms
}

CREATE_TARGET_TABLE_QUERY = f"""
    CREATE OR REPLACE TABLE {TARGET_DATABASE}.{TARGET_SCHEMA}.{TARGET_TABLE} (
        invocation_id        VARCHAR,
        project_id           VARCHAR,
        generated_at         TIMESTAMP_NTZ,
        unique_id            VARCHAR,
        fqn                  VARCHAR,
        resource_type        VARCHAR,
        materialization_type VARCHAR,
        database             VARCHAR,
        schema               VARCHAR,
        model_name           VARCHAR,
        package_name         VARCHAR,
        relation_name        VARCHAR
    )
"""

In [ ]:
def fetch_manifests_from_presigned_urls(
        file_name: str, 
        approx_manifest_count: int, 
        margin: int
    ) -> tuple[
        Optional[list[str]], 
        Optional[list[str]]
    ]:
    """
    Clean up urls from txt file into list, loop through list of urls
    and write content to json files in data/ dir in the format of
    `manifest-{loop-iteration}.json`

    Any malformed links or links that fail to authenticate for
    any reason will just be ignored and the exception will be added
    to a list, mostly just to profile how many fail, not a big deal
    if this dataset is incomplete for now

    params:
        file_name: str -> name of .txt file that contains \n separated
            presigned s3 urls; these urls have a short TTL so it is preferred
            to request as many as possible and download the manifest files to
            be worked up before trying to process any of the manifests

        approx_manifest_count: int -> rough count of the expected number of
            manifest.json files to be downloaded; used to alert when actual
            number of files is considerably less than expected and for rough
            idempotency -- see params.margin for idempotency considerations

        margin: int -> the acceptable margin to avoid re-running this method;
            if this method has been run before files will already exist in `data`
            dir, so if actual file count - params.approx_manifest_count < margin
            this method will perform a no-op rather than re-request files

    returns:
        Currently the objects returned by this method aren't used downstream
        but could be useful for debugging; the useful output of this method
        is a side-effect - manifest.json files written to TARGET_DIR

        urls: list[str] -> list of url strings that were passed to urllib3
        e_log: list[str] -> list of exceptions encountered when trying to 
            send GET request to url in urls list
    """
    files_present = len(list(os.scandir(TARGET_DIR)))

    if files_present > (approx_manifest_count - margin):
        print("It looks like you might have already run this method...")
        print(f"""
            Expected manifests: ~{approx_manifest_count}
            Present files in target dir: {files_present}
            Margin of Error: {margin}

            You may have executed this method in a different session, but
            check {TARGET_DIR} directory for manifest.json files, otherwise
            you run the risk of re-exporting manifests that are already
            present which could lead to duplicate data.

            If there are many files present but fewer than you expected 
            you may want to clear out {TARGET_DIR} and rerun this method,
            potentially with adjusted params, and then inspect the returns
            to determine of unexpected errors are occuring in the url parsing.

            Common errors could include:
              - expired tokens on the presigned urls
              - malformed urls
                - incorrectly copied or incomplete
                - two urls that aren't separated by \n
                - erroneous or additional newlines
        """)
        return None, None 
    else:
        with open(f'{PRESIGNED_URLS_TXT_FILE_NAME}.txt') as f:
            lines = f.readlines()

        # Probably could clean this up in the open() call, but
        # every line comes with '\n', so a quick loop to remove
        # the last character preps the url for request
        urls = []

        for line in lines:
            urls.append(line[:-1])

        http = urllib3.PoolManager()

        i = 1
        e_log = []
        for url in urls:
            try:
                filename = f'data/manifest-{i}.json'
                with http.request('GET',url, preload_content=False) as resp, open(filename, 'wb') as out_file:
                    shutil.copyfileobj(resp, out_file)
                i += 1
            except Exception as e:
                e_log.append(e)

        return urls, e_log

It looks like you might have already run this method...

            Expected manifests: ~2000
            Present files in target dir: 1875
            Margin of Error: 250

            You may have executed this method in a different session, but
            check data directory for manifest.json files, otherwise
            you run the risk of re-exporting manifests that are already
            present which could lead to duplicate data.

            If there are many files present but fewer than you expected 
            you may want to clear out data and rerun this method,
            potentially with adjusted params, and then inspect the returns
            to determine of unexpected errors are occuring in the url parsing.

            Common errors could include:
              - expired tokens on the presigned urls
              - malformed urls
                - incorrectly copied or incomplete
                - two urls that aren't separated by 

                - erroneous or

In [ ]:
def parse_manifests(
        dir: str, 
        file_warn: int = 10000, 
        force: bool = False
    ) -> tuple[
        Optional[list[dict]], 
        Optional[list[str]], 
        Optional[list[str]]
    ]:
    """
    Opens each manifest file in target directory and processes
    the data as JSON. Metadata and node data are extracted from
    relevant nested structs within the manifest.json files into
    flat python dictionaries.

    Each dictionary is appended to a list of dictionaries which
    can be used downstream to build a dataframe where each dictionary
    represents a row in the resultant data structure

    params:
        dir: str -> string name of the directory that the manifest.json
            files are contained within

        file_warn: int -> number of files that might start causing mem
            or processing time issues; easily overridable, but set to
            10,000 by default based on computation time testing on a
            MacBook Pro (M5)

        force: bool -> Can be used to brute force past the warning logic

    returns:
        Currently only the first list returned by this method is used
        downstream, but the other two are lists of exceptions that may
        have occured during the process of opening a file or accessing
        the `nodes` object within the file.

        rows_to_insert: list[dict] -> list of dictionaries that can be
            inserted into a dataframe (Polars, Pandas, etc)
        node_e_log: list[str] -> list of exceptions encountered when trying
            to extract dictionary data from `nodes` struct in JSON file
        file_e_log: list[str] -> list of exceptions encountered when trying
            to open or access a file
    """

    files = list(os.scandir(dir))
    file_count = len(files)

    if file_count > file_warn and not force:
        print(f"""
            Whoa there big dawg, that's a lot of files... {file_count}!

            Every 1000 files takes approximately 2 minutes to process
            and could take up quite a bit of memory (up to 400MB per file).

            You can rerun this bad boy with a higher than default warning limit
            (which is currently set to 10,000) or with Force = True to
            circumvent this warning, but just be sure you know what you're doing.

            It would suck to kick this code off only to wait 1-2H, come back, 
            and have nothing because you ran out of memory. Consider chunks.
        """)
        return None, None, None

    else:
        rows_to_insert = []

        node_e_log = []
        file_e_log = []

        for file in files:
            with open(file.path, 'r') as manifest:
                try:
                    data = json.load(manifest)

                    invocation_id = data["metadata"]["invocation_id"]
                    project_id = data["metadata"]["project_id"]
                    generated_at = data["metadata"]["generated_at"]
                    nodes = data["nodes"]
                    node_list = list(nodes.keys())

                    for node in node_list:
                        try:
                            n = {
                                "invocation_id": invocation_id,
                                "project_id": project_id,
                                "generated_at": generated_at,
                                "unique_id": data["nodes"][node]["unique_id"],
                                "fqn": ".".join(data["nodes"][node]["fqn"]),
                                "resource_type": data["nodes"][node]["resource_type"],
                                "materialization_type": data["nodes"][node]["config"]["materialized"],
                                "database": data["nodes"][node]["database"],
                                "schema": data["nodes"][node]["schema"],
                                "model_name": data["nodes"][node]["name"],
                                "package_name": data["nodes"][node]["package_name"],
                                "relation_name": data["nodes"][node]["relation_name"],
                            }
                        except Exception as e:
                            node_e_log.append({node: e})

                        rows_to_insert.append(n)

                except Exception as e:
                    file_e_log.append({file: e})

        return rows_to_insert, node_e_log, file_e_log




In [ ]:
def dicts_to_polars(rows: list[dict]) -> pl.DataFrame: 
    """
    Converts list of dicts into rows of a Polars DataFrame.
    Column names are inferred from dictionary keys (no logic
    is present to resolve the case of mismatched or misordered
    keys / columns, as such this method assumes that every 
    instance in params.rows is an identically structured dict).

    Data types are ALL str in the ingested dictionaries as no
    transformation or coercion is done during manifest processing.

    That said, all but one column is expected to be of str / VARCHAR.
    The only exception is the `generated_at` column which can be
    easily converted from str -> timestamp [utc]

    params:
        rows: list[dict] -> list of dicts containing data extracted
            from manifest.jsons

            expected dictionary structure:
            {
                "invocation_id":        str,
                "project_id":           str,
                "generated_at":         str,
                "unique_id":            str,
                "fqn":                  str,
                "resource_type":        str,
                "materialization_type": str,
                "database":             str,
                "schema":               str,
                "model_name":           str,
                "package_name":         str,
                "relation_name":        str,
            }

    returns:
        df: pl.DataFrame -> Dataframe with schema:
            pl.DataFrame(
                schema={
                    "invocation_id":        str,
                    "project_id":           str,
                    "generated_at":         Datetime(time_unit='us', time_zone='UTC'),
                    "unique_id":            str,
                    "fqn":                  str,
                    "resource_type":        str,
                    "materialization_type": str,
                    "database":             str,
                    "schema":               str,
                    "model_name":           str,
                    "package_name":         str,
                    "relation_name":        str,
                }
            )
    """

    # convert list of dicts to polars df that uses
    # dict.keys() for col names
    df = pl.DataFrame(rows_to_insert)

    # datatypes are all str when pulling data from JSON file
    # correct generated_at from str -> datetime (nanosecond)
    # afaik generated_at is written as UTC, so we will enforce
    # that in the type conversion as required by polars
    # see: https://github.com/search?q=repo%3Adbt-labs%2Fdbt+generated_at+timezone&type=code
    df = df.with_columns(
        # select the col to transform it but aliasing it back to the org name just
        # overwrites the original col
        pl.col("generated_at").str.to_datetime(time_zone="UTC").alias("generated_at")
    )

    return df



In [ ]:
def _preview_data(df: pl.DataFrame, display_width: int = 250):
    """
    Lightweight wrapper around pl.Config and pl.DataFrame.show() that makes
    the output more human readable and adjustable to any given display.

    tbl_cols = -1 ensures that the dataframe isn't truncated and that all
        cols are displayed

    params:
        df: pl.DataFrame -> Any polars DataFrame
        display_width: int -> int value for pixel width at which df should be displayed
            defaults to 250, seems to work well for a 16" MacBook Pro.
    """
    # display dataframe with customizable width and all columns
    with pl.Config(fmt_str_lengths=display_width, tbl_width_chars=display_width, tbl_cols=-1):
        df.show()

invocation_id,project_id,generated_at,unique_id,fqn,resource_type,materialization_type,database,schema,model_name,package_name,relation_name
str,str,"datetime[μs, UTC]",str,str,str,str,str,str,str,str,str
"""1c61b183-590f-43ad-8262-a1d7c2…","""04e365cf153278183f45e008e58b87…",2026-10-02 14:37:16.533927 UTC,"""model.rdl.in_ps_pln_sls_contri…","""rdl.staging.adls.in_ps_pln_sls…","""model""","""incremental""","""RDL_RAW""","""landing""","""in_ps_pln_sls_contrib_raw""","""rdl""","""RDL_RAW.landing.in_ps_pln_sls_…"
"""1c61b183-590f-43ad-8262-a1d7c2…","""04e365cf153278183f45e008e58b87…",2026-10-02 14:37:16.533927 UTC,"""model.rdl.in_sco_stores_raw""","""rdl.staging.adls.in_sco_stores…","""model""","""incremental""","""RDL_RAW""","""landing""","""in_sco_stores_raw""","""rdl""","""RDL_RAW.landing.in_sco_stores_…"
"""1c61b183-590f-43ad-8262-a1d7c2…","""04e365cf153278183f45e008e58b87…",2026-10-02 14:37:16.533927 UTC,"""model.rdl.sat_shipment_trackin…","""rdl.refined_vault.shipment.sat…","""model""","""incremental""","""RDL_REFINED_VAULT_DEV""","""shipment""","""sat_shipment_tracking_acs_info…","""rdl""","""RDL_REFINED_VAULT_DEV.shipment…"
"""1c61b183-590f-43ad-8262-a1d7c2…","""04e365cf153278183f45e008e58b87…",2026-10-02 14:37:16.533927 UTC,"""model.rdl.sat_shipment_plannin…","""rdl.refined_vault.shipment.sat…","""model""","""incremental""","""RDL_REFINED_VAULT_DEV""","""shipment""","""sat_shipment_planning_detail_a…","""rdl""","""RDL_REFINED_VAULT_DEV.shipment…"
"""1c61b183-590f-43ad-8262-a1d7c2…","""04e365cf153278183f45e008e58b87…",2026-10-02 14:37:16.533927 UTC,"""model.rdl.sat_shipment_event_d…","""rdl.refined_vault.shipment.sat…","""model""","""incremental""","""RDL_REFINED_VAULT_DEV""","""shipment""","""sat_shipment_event_dhl_inforne…","""rdl""","""RDL_REFINED_VAULT_DEV.shipment…"


In [29]:
def _create_snowpark_session(connection_parameters: dict[str] = SNOW_CONN_PARAMS):
    """
    Passes dictionary of parameters to Snowflake to establish a reusable
    connection with the desired Snowflake instance.

    params:
        connection_parameters: dict[str] -> Dictionary of named string values
            that correspond to required connection details to reach Snowflake

            example:
            {
                "account": (Snowflake account identifier),
                "user": (Username exactly as it appears in Snowflake),
                "authenticator": (password, externalbrowser, etc),
                "database": (Snowflake database that should be used by default),
                "schema": (Snowflake schema that should be used by default),
                "warehouse": (Snowflake warehouse that should be used for compute), 
                "role": (Snowflake role that your user should assume for processing),     
            } 

            Default SNOW_CONN_PARAMS is currently configured with
                -  account -> dbt internal analytics account
                -  user -> needs to be overwritten by each user that executes this notebook
                -  authenticator = externalbrowser -> uses SSO so no secrets have to be passed
                -  database -> since this is adhoc, so any non-production database should do
                -  schema -> sinec this is adhoc, any schema should do
                -  warehouse -> uses a small warehouse, can up the size if processing is too long
                -  role -> TRANSFORMER is the current default that plays nicely with Snowflake RBAC

            IMPORTANT -- These connection parameters ASSUME that all references to Snowflake objects
                (e.g. user, database, schema, warehouse, role) have been validated as objects that
                exist within the target Snowflake account. If they do not, logic will need to be
                introduced to create the relevant objects (e.g. CREATE DATABASE, CREATE SCHEMA)
    
    returns:
        session: snowpark.Session -> Reusable connection object that allows
            downstream interaction with Snowflake instance
    """
    return Session.builder.configs(connection_parameters).create()


def _create_target_table_in_snowflake(session: Session, create_table_query: str = CREATE_TARGET_TABLE_QUERY):
    """
    Takes a valid Snowpark Sesssion and a predefined Snowflake compatible query in the form of a str;
    executes query string against existing session.

    In theory, this isn't the safest way to execute a query against the Snowflake instance, especially
    if a the default CREATE_TARGET_TABLE_QUERY is edited or hijacked, or a different and unvalidated
    query is passed. But, this should do the trick for this adhoc workflow for now.

    params:
        session: snowpark.Session -> Reusable connection object that allows
            downstream interaction with Snowflake instance
    
        create_table_query: str -> SQL query with the intent of creating or replacing the
            target table for the data extracted from dbt manifest.jsons. If you don't have
            strong reasoning otherwise, you should not pass your own value here and just
            use the predefined constant. Logic will check the query to ensure that it:
                1. only executes ONE query (as determined by ; separated strings)
                2. that query contains (CREATE OR REPLACE TABLE)
            
            If more than one query is present or if the query does not CREATE OR REPLACE TABLE
            this method will exit.
    
    returns:
        ctas_run: list[Row] -> Snowpark object that can be used to validate successful
            (or unsuccessful) query run. Technically a list of rows from snowflake, but
            the output of a CREATE OR REPLACE TABLE query isn't a "row" necessarily, but
            a string output to the console that tells you whether the operation succeeded.
        
    """
    if len(create_table_query.split(";")) > 1:
        print("""
            Hey there buddy, let's make sure we're only trying to execute one query at a time.
            If your query is just that, feel free to remove the ; as it is unnecessary for
            running one statement.
        """)
    elif not create_table_query.strip().startswith("CREATE OR REPLACE TABLE"):
        print("""
            Okay, I know this isn't the safest SQL practice, but please don't use this
            method to do anything but create or replace a table. If you've made it this
            far and are trying to do something that isn't expected by this notebook, or
            worse, nefarious, I'm sure you can find a better way to do it.
        """)
    else:
        ctas_run = session.sql(create_table_query).collect()
        return ctas_run


def _convert_pl_df_to_snowpark_df(session: Session, df: pl.DataFrame):
    """
    Snowflake allows data be ingested in the form of a DataFrame, however
    Snowpark has it's own internal DataFrame structure that is used for 
    type safety purposes. As such, it is necessary to convert from the
    type of DataFrame being worked with in Python (assumed polars) to
    the Snowpark represenation.

    Snowpark DataFrames can be created from other DataFrames with a 
    built in method, however, it only supports pandas DataFrames, so
    a conversion from pl.DataFrame -> pd.DataFrame is necessary before
    calling snowpark.Session.create_dataframe(pd.DataFrame).

    To convert form pl -> pd it is obviously necessary to bring the pd
    lib into scope, however, less obviously it is also necessary to 
    bring pyarrow (pa) into scope because the conversion logic in polars
    depends on using arrow / pyarrow to safely convert to pandas.

    Since these libs are ONLY used for conversion they will be imported
    only into this method's scope.

    params:
        session: snowpark.Session -> Reusable connection object that allows
            downstream interaction with Snowflake instance
        
        df: pl.DataFrame -> Polars DataFrame that contains the data to be
            ingested into Snowflake
         
    """
    import pandas as pd
    import pyarrow as pa
    snowpark_df = session.create_dataframe(df.to_pandas())
    return snowpark_df


In [ ]:
def upload_to_sf(df, mode: str = "insert", table: str = TARGET_TABLE):
    """
    Calls helper methods to create a snowpark.Session, create a target table for data to be ingested to,
    and convert the local DataFrame containing processed manifest.json data to a Snowpark compatible
    DataFrame before ultimately committing that data to Snowflake with some options available to
    dictate how that data is loaded.

    params:
        df: pl.DataFrame -> Polars DataFrame that contains extracted manifest.json data.
            It is expected that the schema of this DataFrame will match the schema of
            the target table in Snowflake, however, currently no logic exists to validate
            that expectation or handle failures for when it is not met.
        mode: str -> Method in which data will be written to Snowflake. "insert" is the default
            which will NAIVELY insert data into the target table even if identical rows already
            exist; since this is an adhoc and notebook-based workflow, currently, that should
            be okay -- of course, retriggering the method, accidentally or otherwise, will
            duplicate the data.

            "overwrite" is also accepted, which will do just that: overwrite ALL of the data
            present in the target table with the contents of the provided DataFrame. Confirmation
            of overwrite is required just in case someone thinks it can be used as an upsert. 
            It cannot.

            Snowpark exposes an `update` method that could be used to emulate an "upsert" like
            flow, however, the adhoc nature of this workload does not necessitate that level of
            over-engineering at this stage.

        table: str -> Name of the table (excluding database.schema) that should serve as the
            target for the DataFrame to be loaded into. Defaults to TARGET_TABLE defined in
            the constants, and as such will play nicely with the Snowflake connection params
            and the name of the table passed along to _create_target_table_in_snowflake().
    """
    session = _create_snowpark_session()
    ctas_result = _create_target_table_in_snowflake(session)
    print(ctas_result)
    sdf = _convert_pl_df_to_snowpark_df(session, df)

    if mode == "insert":
        sdf.write.insert_into(table, overwrite=False)
    elif mode == "overwrite":
        print(f"MAKE SURE YOU REALLY WANT TO OVERWRITE {table}...")
        print(f"Input y if you want to replace all existing data in {table} with local df...")
        print(f"Input n if you want to stop and reconsider...")
        y_or_n = input()

        if y_or_n == "y":
            sdf.write.mode(mode).save_as_table(table)
        elif y_or_n == "n":
            return
        else:
            print(f"Did not understand input: {y_or_n}, please only respond with 'y' or 'n'...")
            return
    else:
        print(f"Did not understand mode: {mode}... please rely on defaults if you are unsure....")

In [ ]:
urls, e_log = fetch_manifests_from_presigned_urls(
    PRESIGNED_URLS_TXT_FILE_NAME, 
    APPROX_MANIFEST_COUNT, 
    MARGIN_OF_ERROR
)

rows_to_insert, node_errors, file_errors = parse_manifests(TARGET_DIR)

df = dicts_to_polars(rows_to_insert)

# _preview_data(df)

upload_to_sf(df)

 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://dbtlabs.okta.com/app/snowflake/exkc5f49sIXGnu9Zk4x6/sso/saml?SAMLRequest=jZJBc9owEIX%2Fikc9Y9mOgVgDZGg8tJ4QygSSNrkJSxCBLBmtHJN%2FX9kOnfSQTG6a1XvSt%2Ft2dHUqpPfCDQitxij0A%2BRxlWsm1G6M7tez3iXywFLFqNSKj9ErB3Q1GQEtZEmmlX1Wd%2FxYcbCee0gBaS%2FGqDKKaAoCiKIFB2JzsprezknkB6Q02upcS%2FTO8rmDAnBjHeHZwkA4vGdrS4JxXdd%2BfeFrs8NREAQ4SLBTNZJvZ%2F3J9fSBPsRB3OidwsmXb2zfhepG8BnWphMB%2BbleL3vLX6s18qZn1GutoCq4WXHzInJ%2BfzfvAMAR3PyOhsN%2B5IPS9VbSA891UVbWveW7E95yhqXeCddulo5ReRBMbkKgRpt0EZkt7KPHhO1VOmD74dyUN1TPbncLpo7ZMX3MkfdwzjNq8swAKp6pJkXrSkE06IVBL0jW4SXpJyRO%2FP5F%2FIS81KUoFLWt84zKNlbSDfj6YGkLR8sS%2F%2BPG%2FHTI%2B9s4gezPD1UlT4f4NMAAGjehom5PSAtgJl%2FrfoTfe94WbeFmn6VLLUX%2B6s20Kaj9OJrQD9uKYL1tKyW8oEJOGTMcwEUkpa6vDafW7bM1FUd40v36%2F0ZP%2FgI%3D&RelayState=ver%3A3-hint%3A15844559260714-ETMsDgAAAaEiCS6eABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEHd%2F7gPFSGNgRBs5ge9AsAsAAACQFXZ8fKEBbEQ4jXhrkTR3eRqAQCh0cpWyK

 pip install snowflake-connector-python[secure-local-storage]


[Row(status='Table MODEL_ID_TO_FQN_MAPPINGS_FROM_MANIFESTS successfully created.')]
